# Vision-LLM 카메라 간편 실행

Jetson CSI 카메라의 YOLO 탐지 결과를 자연어 Context로 바꾸어 Gemma에 전달합니다. **위에서부터 셀을 하나씩 실행**하세요.

- 카메라 창에서 `l`: 현재 탐지 결과를 LLM에 전달
- 카메라 창에서 `q`: 종료
- LLM 생성 중에는 카메라 화면이 잠시 멈출 수 있습니다.
- 이 노트북에서는 메모리 안정성을 위해 Gemma를 CPU로 실행합니다.

## 1. 라이브러리와 모델 준비

이 셀은 YOLO와 Gemma를 각각 한 번만 로드합니다. 모델 로딩에는 시간이 걸릴 수 있습니다.

In [1]:
import cv2
from ultralytics import YOLO
from llama_cpp import Llama


YOLO_MODEL_PATH = "src/models/YOLO/yolo11n_int8.engine"
GEMMA_MODEL_PATH = "src/models/Gemma4/google_gemma-4-E2B-it-Q4_K_M.gguf"
CONTEXT_WINDOW = 1024  # 메모리 사용량을 줄이기 위한 컨텍스트 길이
MAX_TOKENS = 80  # 두 문장 답변에 충분한 생성 길이


yolo = YOLO(YOLO_MODEL_PATH)  # TensorRT YOLO 모델 로드

llm = Llama(
    model_path=GEMMA_MODEL_PATH,
    n_gpu_layers=0,  # Jetson 공유 메모리 부족을 피하기 위해 CPU에서 추론
    n_ctx=CONTEXT_WINDOW,
    n_batch=8,
    n_ubatch=8,
    verbose=False,
)

print("YOLO와 Gemma 모델 준비 완료")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


YOLO와 Gemma 모델 준비 완료


## 2. Vision Context 변환 함수 준비

In [2]:
def result_to_vision_data(result, frame):
    """YOLO 결과를 이미지 크기와 객체 목록을 가진 딕셔너리로 변환합니다."""
    height, width = frame.shape[:2]
    objects = []

    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()  # GPU 좌표를 Python 목록으로 변환
        objects.append(
            {
                "class": result.names[class_id],
                "confidence": round(confidence, 3),
                "bbox": {
                    "x1": int(x1),
                    "y1": int(y1),
                    "x2": int(x2),
                    "y2": int(y2),
                },
            }
        )

    return {"image_width": width, "image_height": height, "objects": objects}


def detections_to_text(vision_data):
    """구조화된 객체 탐지 결과를 LLM이 읽을 자연어로 변환합니다."""
    objects = vision_data["objects"]

    if not objects:
        return "현재 탐지된 객체가 없습니다."

    return "\n".join(
        f"{index}번 객체는 {obj['class']}이며, confidence는 {obj['confidence']:.2f}입니다."
        for index, obj in enumerate(objects, start=1)
    )


def ask_llm(vision_text):
    """Vision Context를 Gemma에 전달하고 답변 문자열을 반환합니다."""
    prompt = f"""
Instruction:
주어진 객체 탐지 정보를 바탕으로 현재 상황을 설명하세요.

Context:
{vision_text}

Constraint:
탐지 결과에 없는 객체를 추측하지 마세요.

Output Format:
한국어 두 문장 이내
"""

    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_TOKENS,
        temperature=0.2,
    )
    return response["choices"][0]["message"]["content"].strip()


print("Vision Context 변환 함수 준비 완료")

Vision Context 변환 함수 준비 완료


## 3. 카메라 실행

셀 실행 후 카메라 창을 한 번 클릭해 키보드 포커스를 준 다음 `l` 또는 `q`를 누르세요. 답변은 아래 셀 출력에 표시됩니다.

In [3]:
CAMERA_INDEX = 0  # 두 번째 CSI 카메라는 1로 변경
CAMERA_WIDTH = 1280
CAMERA_HEIGHT = 720
CAMERA_FPS = 30


camera_pipeline = (
    f"nvarguscamerasrc sensor-id={CAMERA_INDEX} ! "
    f"video/x-raw(memory:NVMM), width={CAMERA_WIDTH}, height={CAMERA_HEIGHT}, "
    f"framerate={CAMERA_FPS}/1 ! "
    "nvvidconv ! "
    "video/x-raw, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "  # OpenCV가 사용할 BGR 프레임으로 변환
    "queue leaky=downstream max-size-buffers=1 ! "
    "appsink drop=true max-buffers=1 sync=false"
)

camera = cv2.VideoCapture(camera_pipeline, cv2.CAP_GSTREAMER)

if not camera.isOpened():
    raise RuntimeError(
        f"CSI 카메라 sensor-id={CAMERA_INDEX}를 열 수 없습니다. "
        "카메라 연결과 nvargus-daemon 상태를 확인하세요."
    )

print("카메라 창에서 [l] LLM 질문 / [q] 종료")

try:
    while True:
        success, frame = camera.read()  # CSI 카메라의 최신 프레임 읽기
        if not success:
            print("카메라 프레임을 읽지 못했습니다.")
            break

        result = yolo.predict(
            source=frame,
            conf=0.25,
            iou=0.5,
            verbose=False,
        )[0]
        annotated_frame = result.plot()  # 탐지 상자와 클래스 이름 표시

        cv2.putText(
            annotated_frame,
            "[l] Ask LLM  [q] Quit",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2,
        )
        cv2.imshow("Vision-LLM Camera", annotated_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break

        if key == ord("l"):
            vision_data = result_to_vision_data(result, frame)
            vision_text = detections_to_text(vision_data)
            print("\n[Vision Context]")
            print(vision_text)
            print("\nGemma 응답 생성 중...")
            answer = ask_llm(vision_text)  # 생성 중 카메라 화면이 잠시 멈출 수 있음
            print("\n[Gemma]")
            print(answer)

finally:
    camera.release()  # 어떤 종료 상황에서도 카메라 반환
    cv2.destroyAllWindows()
    print("카메라를 종료했습니다.")

GST_ARGUS: Creating output stream
CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3280 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3280 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: Running with following settings:
   Camera index = 0 
   Camera mode  = 4 
   Output Stream W = 1280 H = 7

[ WARN:0@24.512] global cap_gstreamer.cpp:1728 open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


[08/14/2026-15:33:58] [TRT] [I] Loaded engine size: 4 MiB
[08/14/2026-15:33:58] [TRT] [W] Using an engine plan file across different models of devices is not recommended and is likely to affect performance or even cause errors.
[08/14/2026-15:33:58] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +9, now: CPU 0, GPU 12 (MiB)


Gtk-Message: 15:34:00.960: Failed to load module "canberra-gtk-module"


GST_ARGUS: Cleaning up
CONSUMER: Done Success
GST_ARGUS: Done Success
카메라를 종료했습니다.


## 문제 해결

카메라가 열리지 않으면 노트북의 모든 카메라 셀을 중지한 뒤 터미널에서 다음을 실행하세요.

```bash
sudo systemctl restart nvargus-daemon
```

커널이 종료되면 커널을 재시작하고 이 노트북만 위에서부터 다시 실행하세요. 이미지 기반 Vision-Language 노트북은 동시에 실행하지 않는 것이 좋습니다.